# ETL (Extraction, Transformation, Loading)
## IBEX 35 Investment Simulator

En este notebook vamos a realizar el proceso de obtención de datos para nuestro proyecto. Nuestro flujo de trabajo será:
1. **Extracción**: Obtener los componentes actuales del IBEX 35 desde Wikipedia (Tickers y Sectores).
2. **Descarga**: Obtener precios históricos de los últimos 5 años usando `yfinance`.
3. **Limpieza**: Tratar valores nulos y asegurar la integridad de las series temporales.
4. **Carga**: Preparar el dataset final (CSV) para nuestro modelo SQL.

In [70]:
import pandas as pd
import requests
import yfinance as yf
import time
import re

print("✅ Librerías preparadas para nuestro análisis.")

✅ Librerías preparadas para nuestro análisis.


### Scraping de Wikipedia (Tickers y Sectores)

**Nuestra estrategia:**
- Accedemos a la tabla de componentes del IBEX 35 en Wikipedia.
- Utilizamos un `User-Agent` en nuestra petición para identificarnos correctamente como un navegador.
- Convertimos el HTML en DataFrames de Pandas para manipularlos fácilmente.
- Seleccionamos la tabla de componentes y preparamos los tickers con el formato que requiere Yahoo Finance.

In [71]:
# Configuramos nuestra petición
url = 'https://es.wikipedia.org/wiki/IBEX_35'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko)'}

# Obtenemos los datos y extraemos las tablas
response = requests.get(url, headers=headers)
tablas = pd.read_html(response.text)

# Localizamos la tabla de componentes (índice 1)
df_ibex_raw = tablas[1]
display(df_ibex_raw.head(35))

print(f"✅ Hemos extraído correctamente {len(df_ibex_raw)} empresas.")

C:\Users\manue\AppData\Local\Temp\ipykernel_31096\1541135219.py:7: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tablas = pd.read_html(response.text)


,Ticker,Empresa,Sede,Entrada,Sector[30]​,ISIN,Ponderación (Sep. 2022)
0,ANA,Acciona,Alcobendas,2015,Construcción,ES0125220311,112
1,ANE,Acciona Energía,Alcobendas,2022,Energías renovables,ES0105563003,36
2,ACX,Acerinox,Madrid,2015,"Mineral, metales y transformación",ES0132105018,49
3,ACS,ACS,Madrid,1998,Construcción,ES0167050915,203
4,AENA,Aena,Madrid,2015,Transporte y distribución,ES0105046009,351
5,AMS,Amadeus IT Group,Madrid,2011,Electrónica y software,ES0109067019,519
6,MTS,ArcelorMittal,Ciudad de Luxemburgo,2009,"Mineral, metales y transformación",LU1598757687,76
7,SAB,Banco Sabadell,Sabadell,2004,Bancos y cajas de ahorro,ES0113860A34,141
8,BKT,bankinter.,Madrid,1992,Bancos y cajas de ahorro,ES0113679I37,115
9,BBVA,BBVA,Bilbao,1992,Bancos y cajas de ahorro,ES0113211835,948


✅ Hemos extraído correctamente 35 empresas.


### Transformación y Normalización

**Lo que estamos haciendo para dejar los datos impecables:**
- Añadimos el sufijo `.MC` a cada ticker.
- Renombramos todas las columnas a **minúsculas** y simplificamos 'sector'.
- Limpiamos la columna `sector` eliminando números o referencias.
- Generamos nuestra lista final de trabajo.

In [72]:
# Añadimos el sufijo para Yahoo Finance
df_ibex_raw['Ticker_Yahoo'] = df_ibex_raw['Ticker'].apply(lambda x: f"{x.strip()}.MC")

# Filtramos las columnas originales (usando los nombres de Wikipedia)
df_componentes = df_ibex_raw[['Ticker_Yahoo', 'Empresa', 'Sector[30]​']].copy()

# Normalizamos y simplificamos los nombres de las columnas a minúsculas
df_componentes.columns = ['ticker_yahoo', 'empresa', 'sector']

# Limpiamos la columna 'sector' eliminando cualquier número [0-9]
df_componentes['sector'] = df_componentes['sector'].str.replace(r'\d+', '', regex=True).str.strip()

# 1. Eliminamos espacios en blanco y el punto final 
df_componentes['empresa'] = df_componentes['empresa'].str.strip().str.rstrip('.')

# Usamos replace para poner bien Bankinter
df_componentes['empresa'] = df_componentes['empresa'].replace({'bankinter': 'Bankinter'})
# Lista final de tickers
tickers_ibex = df_componentes['ticker_yahoo'].tolist()

print("Nuestra lista de activos normalizada:")
print(tickers_ibex)

df_componentes.head(30)

Nuestra lista de activos normalizada:
['ANA.MC', 'ANE.MC', 'ACX.MC', 'ACS.MC', 'AENA.MC', 'AMS.MC', 'MTS.MC', 'SAB.MC', 'BKT.MC', 'BBVA.MC', 'CABK.MC', 'CLNX.MC', 'ENG.MC', 'ELE.MC', 'FER.MC', 'FDR.MC', 'GRF.MC', 'IAG.MC', 'IBE.MC', 'ITX.MC', 'IDR.MC', 'COL.MC', 'LOG.MC', 'MAP.MC', 'MEL.MC', 'MRL.MC', 'NTGY.MC', 'RED.MC', 'REP.MC', 'ROVI.MC', 'SCYR.MC', 'SAN.MC', 'SLR.MC', 'TEF.MC', 'UNI.MC']


,ticker_yahoo,empresa,sector
0,ANA.MC,Acciona,Construcción
1,ANE.MC,Acciona Energía,Energías renovables
2,ACX.MC,Acerinox,"Mineral, metales y transformación"
3,ACS.MC,ACS,Construcción
4,AENA.MC,Aena,Transporte y distribución
5,AMS.MC,Amadeus IT Group,Electrónica y software
6,MTS.MC,ArcelorMittal,"Mineral, metales y transformación"
7,SAB.MC,Banco Sabadell,Bancos y cajas de ahorro
8,BKT.MC,Bankinter,Bancos y cajas de ahorro
9,BBVA.MC,BBVA,Bancos y cajas de ahorro


### Descarga de Históricos

**Nuestra estrategia de descarga:**
- Definimos una ventana de tiempo de **5 años** para tener datos suficientes para validar nuestras hipótesis de volatilidad y sectores.
- Usamos `yf.download` para realizar una descarga masiva y eficiente.
- Además de los 35 activos del IBEX, incluimos tres puntos de referencia (benchmarks):
  - `^IBEX`: El mercado español para compararnos.
  - `^VIX`: El índice de volatilidad (el "medidor del miedo") global.
  - `GC=F`: El contrato de futuros del Oro (activo refugio habitual en crisis).
- Nos centraremos en el **Precio de Cierre Ajustado (Adj Close)**, ya que refleja la rentabilidad real incluyendo dividendos y splits.

In [73]:
# Preparamos la lista extendida de tickers
benchmarks = ['^IBEX', '^VIX', 'GC=F']
tickers_totales = tickers_ibex + benchmarks

print(f"Vamos a descargar datos para {len(tickers_totales)} activos.")

# Ejecutamos la descarga masiva

df_raw = yf.download(tickers_totales, period='5y', auto_adjust=False)

# Seleccionamos las columnas relevantes
df_precios = df_raw[['Adj Close', 'Volume', 'High', 'Low']].copy()

print("\nDescarga completada. Estructura de nuestros datos:")
display(df_precios.head())

Vamos a descargar datos para 38 activos.


[*********************100%***********************]  38 of 38 completed



Descarga completada. Estructura de nuestros datos:


Price       Adj Close                                                     \
Ticker         ACS.MC    ACX.MC    AENA.MC     AMS.MC      ANA.MC ANE.MC   
Date                                                                       
2021-03-12  19.781284  7.655886  12.806684  59.835766  111.493767    NaN   
2021-03-15  20.183643  7.630121  12.921901  58.252705  115.916092    NaN   
2021-03-16  20.278748  7.644844  12.930763  58.497704  115.660957    NaN   
2021-03-17  20.000753  7.530742  12.780097  57.875786  115.065643    NaN   
2021-03-18  20.300694  7.714777  12.753510  58.422325  114.130157    NaN   

Price                                                ...     Low             \
Ticker       BBVA.MC    BKT.MC   CABK.MC    CLNX.MC  ...  REP.MC    ROVI.MC   
Date                                                 ...                      
2021-03-12  3.573779  3.094937  1.866038  39.422005  ...  10.720  42.700001   
2021-03-15  3.516897  3.045071  1.862318  40.548603  ...  10.820  42.900002   
2021-03-16  3.498187  3.029894  1.906216  41.984112  ...  10.615  43.299999   
2021-03-17  3.570785  3.085180  1.915144  41.793320  ...  10.625  44.299999   
2021-03-18  3.640389  3.117702  1.974667  41.111900  ...  10.610  44.299999   

Price                                                                        \
Ticker      SAB.MC  SAN.MC   SCYR.MC     SLR.MC TEF.MC  UNI.MC        ^IBEX   
Date                                                                          
2021-03-12  0.4503  2.9000  2.024717  18.500000  3.921  0.7645  8554.400391   
2021-03-15  0.4601  2.9010  2.087076  17.889999  4.011  0.7640  8617.599609   
2021-03-16  0.4663  2.8870  2.090974  18.190001  4.051  0.7710  8617.799805   
2021-03-17  0.4691  2.8950  2.081230  17.600000  4.035  0.7640  8570.099609   
2021-03-18  0.4855  2.9445  2.075384  17.600000  4.054  0.7800  8587.099609   

Price                  
Ticker           ^VIX  
Date                   
2021-03-12  20.629999  
2021-03-15  19.870001  
2021-03-16  19.330000  
2021-03-17  19.180000  
2021-03-18  18.950001  

[5 rows x 152 columns]

### Paso 4: Limpieza y Gestión de Valores Nulos


- Es muy común tener valores nulos en series bursátiles por días festivos o activos que empezaron a cotizar más tarde.
- **Estrategia**: 
  1. Contaremos cuántos nulos tenemos por activo para identificar si alguno tiene lagunas críticas.
  2. Aplicaremos un **Forward Fill (`ffill`)**: Si el lunes no hay precio, usamos el del viernes anterior (es una práctica estándar en finanzas).
  3. Por seguridad, aplicaremos un **Backward Fill (`bfill`)** por si algún activo tiene nulos al inicio de la serie.
- Finalmente, verificaremos que ya no queden valores nulos que puedan romper nuestro análisis estadístico de mañana.

In [74]:
# Vemos cuántos nulos hay 
print("Nulos por columna antes de la limpieza:")
print(df_precios.isnull().sum())

# Limpieza sistemática
# Rellenamos hacia adelante 
df_clean = df_precios.ffill()
# Rellenamos hacia atrás 
df_clean = df_clean.bfill()

# Verificamos el resultado final
print("\n✅ Conteo final de nulos:")
print(df_clean.isnull().sum())
display(df_clean.head())

Nulos por columna antes de la limpieza:
Price      Ticker 
Adj Close  ACS.MC     12
           ACX.MC     12
           AENA.MC    12
           AMS.MC     12
           ANA.MC     12
                      ..
Low        SLR.MC     12
           TEF.MC     12
           UNI.MC     12
           ^IBEX      12
           ^VIX       36
Length: 152, dtype: int64

✅ Conteo final de nulos:
Price      Ticker 
Adj Close  ACS.MC     0
           ACX.MC     0
           AENA.MC    0
           AMS.MC     0
           ANA.MC     0
                     ..
Low        SLR.MC     0
           TEF.MC     0
           UNI.MC     0
           ^IBEX      0
           ^VIX       0
Length: 152, dtype: int64


Price       Adj Close                                                         \
Ticker         ACS.MC    ACX.MC    AENA.MC     AMS.MC      ANA.MC     ANE.MC   
Date                                                                           
2021-03-12  19.781284  7.655886  12.806684  59.835766  111.493767  27.139866   
2021-03-15  20.183643  7.630121  12.921901  58.252705  115.916092  27.139866   
2021-03-16  20.278748  7.644844  12.930763  58.497704  115.660957  27.139866   
2021-03-17  20.000753  7.530742  12.780097  57.875786  115.065643  27.139866   
2021-03-18  20.300694  7.714777  12.753510  58.422325  114.130157  27.139866   

Price                                                ...     Low             \
Ticker       BBVA.MC    BKT.MC   CABK.MC    CLNX.MC  ...  REP.MC    ROVI.MC   
Date                                                 ...                      
2021-03-12  3.573779  3.094937  1.866038  39.422005  ...  10.720  42.700001   
2021-03-15  3.516897  3.045071  1.862318  40.548603  ...  10.820  42.900002   
2021-03-16  3.498187  3.029894  1.906216  41.984112  ...  10.615  43.299999   
2021-03-17  3.570785  3.085180  1.915144  41.793320  ...  10.625  44.299999   
2021-03-18  3.640389  3.117702  1.974667  41.111900  ...  10.610  44.299999   

Price                                                                        \
Ticker      SAB.MC  SAN.MC   SCYR.MC     SLR.MC TEF.MC  UNI.MC        ^IBEX   
Date                                                                          
2021-03-12  0.4503  2.9000  2.024717  18.500000  3.921  0.7645  8554.400391   
2021-03-15  0.4601  2.9010  2.087076  17.889999  4.011  0.7640  8617.599609   
2021-03-16  0.4663  2.8870  2.090974  18.190001  4.051  0.7710  8617.799805   
2021-03-17  0.4691  2.8950  2.081230  17.600000  4.035  0.7640  8570.099609   
2021-03-18  0.4855  2.9445  2.075384  17.600000  4.054  0.7800  8587.099609   

Price                  
Ticker           ^VIX  
Date                   
2021-03-12  20.629999  
2021-03-15  19.870001  
2021-03-16  19.330000  
2021-03-17  19.180000  
2021-03-18  18.950001  

[5 rows x 152 columns]

In [75]:
# 1. Definimos cuántos activos esperamos (35 del IBEX + 3 Benchmarks)
n_activos_esperados = len(df_componentes) + 3
# 2. Verificamos los activos únicos (mirando el nivel de los Tickers en las columnas)
activos_detectados = df_precios.columns.get_level_values(1).unique()
n_activos_reales = len(activos_detectados)
# 3. Verificamos integridad (Solo miramos precios <= 0 en la columna Adj Close)
precios_invalidos = (df_precios['Adj Close'] <= 0).sum().sum()
# 4. Verificamos consistencia comparando listas de Tickers
tickers_sin_precio = [t for t in tickers_ibex if t not in activos_detectados]
print(f"Activos totales detectados: {n_activos_reales} (Esperados: {n_activos_esperados})")
print(f"Periodo: Desde {df_precios.index.min().date()} hasta {df_precios.index.max().date()}")
print(f"Precios <= 0: {precios_invalidos}")
print(f"Activos perdidos en el camino: {len(tickers_sin_precio)}")
if n_activos_reales == n_activos_esperados and precios_invalidos == 0:
    print("\n✅ Datos validados correctamente.")
else:
    print("\n⚠️ ALERTA: Revisa los resultados del informe arriba.")

Activos totales detectados: 38 (Esperados: 38)
Periodo: Desde 2021-03-12 hasta 2026-03-12
Precios <= 0: 0
Activos perdidos en el camino: 0

✅ Datos validados correctamente.


### Paso 5: Carga (Exportación a CSV)

**Nuestro objetivo final del día:**
- Para que mañana podamos hacer el análisis estadístico y el jueves cargar los datos en SQL, necesitamos guardar nuestro trabajo en archivos físicos.
- Vamos a generar dos archivos:
  1. `ibex35_precios.csv`: Contendrá todas las series temporales limpias de los 38 activos.
  2. `ibex35_componentes.csv`: Actuará como nuestra tabla de dimensiones (Ticker, Empresa, Sector).

In [76]:
#  Guardamos la lista de empresas y sectores 
df_componentes.to_csv('../data/ibex35_componentes.csv', index=False)

# Usamos .stack(level=1) para pasar los Tickers de las columnas a las filas
df_export = df_clean.stack(level=1).reset_index()

# Renombramos las columnas para que coincidan exactamente con lo que espera MySQL
df_export.rename(columns={
    'level_1': 'Ticker',
    'Adj Close': 'precio_cierre',
    'Volume': 'volumen',
    'High': 'precio_max',
    'Low': 'precio_min'
}, inplace=True)

# Guardamos los precios históricos limpios en formato LARGO
df_export.to_csv('../data/ibex35_precios.csv', index=False)

print("✅ Archivos guardados correctamente:")
print("- ibex35_componentes.csv")
print("- ibex35_precios.csv (Formato LARGO para MySQL)")


C:\Users\manue\AppData\Local\Temp\ipykernel_31096\2745471558.py:5: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_export = df_clean.stack(level=1).reset_index()


✅ Archivos guardados correctamente:
- ibex35_componentes.csv
- ibex35_precios.csv (Formato LARGO para MySQL)


In [77]:
df_clean.isnull().sum()


Price      Ticker 
Adj Close  ACS.MC     0
           ACX.MC     0
           AENA.MC    0
           AMS.MC     0
           ANA.MC     0
                     ..
Low        SLR.MC     0
           TEF.MC     0
           UNI.MC     0
           ^IBEX      0
           ^VIX       0
Length: 152, dtype: int64